# WNBA Draft Fit Predictor — Validation

This notebook tracks how the 2026 rookie class is actually performing
compared to our predicted Rookie Impact Scores.

Updated as the 2026 WNBA season progresses.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load predictions
predictions = pd.read_csv("../data/processed/predictions_2026.csv")

print("Predicted RIS for 2026 class:")
print(predictions[['player', 'wnba_team', 'draft_pick', 'predicted_RIS']]
      .sort_values('predicted_RIS', ascending=False)
      .to_string())

Predicted RIS for 2026 class:
              player           wnba_team  draft_pick  predicted_RIS
7       Lauren Betts  Washington Mystics           4          0.595
8        Madina Okot       Atlanta Dream          13          0.521
6          Kiki Rice       Toronto Tempo           6          0.496
3   Flau'jae Johnson       Seattle Storm           8          0.486
4    Gabriela Jaquez         Chicago Sky           5          0.481
9       Olivia Miles      Minnesota Lynx           2          0.476
1          Azzi Fudd        Dallas Wings           1          0.414
10     Raven Johnson       Indiana Fever          10          0.333
11        Taina Mair       Seattle Storm          14          0.326
5   Gianna Kneepkens     Connecticut Sun          15          0.301
0     Angela Dugalic  Washington Mystics           9          0.300
2      Cotie McMahon  Washington Mystics          11          0.288


In [8]:
# Load current 2026 WNBA stats
current_stats = pd.read_csv("../data/external/wnba_2026_current_stats.csv")
print(current_stats.shape)
print(current_stats.columns.tolist())
print(current_stats.head())

(166, 28)
['Player', 'Team', 'Pos', 'G', 'MP', 'G.1', 'GS', 'MP.1', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'FT', 'FTA', 'FT%', 'ORB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']
             Player Team  Pos  G   MP  G.1  GS  MP.1   FG   FGA  ...  FTA  \
0    Julie Allemand  TOR    G  2   59    2   2  29.5  1.0   3.5  ...  0.0   
1     Rebecca Allen  NYL  F-G  1   14    1   0  14.0  1.0   3.0  ...  0.0   
2  Laeticia Amihere  GSV    F  3   51    3   0  17.0  2.3   4.0  ...  3.0   
3    Georgia Amoore  WAS    G  2   42    2   2  21.0  2.5   8.5  ...  0.0   
4    Pauline Astier  NYL    G  4  107    4   4  26.8  6.3  10.0  ...  4.3   

     FT%  ORB  TRB  AST  STL  BLK  TOV   PF   PTS  
0    NaN  0.0  3.0  4.0  2.0  0.0  1.5  0.5   2.5  
1    NaN  0.0  0.0  1.0  1.0  0.0  0.0  2.0   3.0  
2  0.556  1.3  3.7  2.3  0.3  1.7  1.3  2.7   6.3  
3    NaN  0.0  2.0  5.0  1.5  0.0  3.0  2.0   6.5  
4  0.706  1.3  3.8  4.0  1.5  0.0  1.8  3.5  16.8  

[5 rows x 28 columns]

In [9]:
# Fix special character names in current stats
current_stats['Player'] = current_stats['Player'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii')

# Check what our rookie names look like vs current stats
print("Our names:", sorted(predictions['player'].tolist()))
print("\nCurrent stats names (filtered):")
name_check = current_stats[current_stats['Player'].str.contains('Dugal|McMahon|Mair', na=False)]
print(name_check[['Player', 'Team', 'G', 'PTS']].to_string())

Our names: ['Angela Dugalic', 'Azzi Fudd', 'Cotie McMahon', "Flau'jae Johnson", 'Gabriela Jaquez', 'Gianna Kneepkens', 'Kiki Rice', 'Lauren Betts', 'Madina Okot', 'Olivia Miles', 'Raven Johnson', 'Taina Mair']

Current stats names (filtered):
            Player Team  G  PTS
49  Angela Dugalic  WAS  2  3.5


In [10]:
# Filter to just our 2026 rookies
rookie_names = predictions['player'].tolist()

# Current stats for our rookies
current_rookies = current_stats[current_stats['Player'].isin(rookie_names)].copy()

print(f"Rookies found in current stats: {len(current_rookies)}")
print(current_rookies[['Player', 'Team', 'G', 'MP.1', 'PTS', 'TRB', 'AST']].to_string())

Rookies found in current stats: 10
               Player Team  G  MP.1   PTS  TRB  AST
12       Lauren Betts  WAS  2  15.0   3.5  3.5  1.0
49     Angela Dugalic  WAS  2  11.5   3.5  2.0  0.0
55          Azzi Fudd  DAL  2  18.5   5.5  1.0  0.0
83    Gabriela Jaquez  CHI  2  29.0   8.5  6.0  2.0
85   Flau'jae Johnson  SEA  3  28.0  11.7  4.7  1.7
86      Raven Johnson  IND  2   8.5   2.0  1.5  2.0
91   Gianna Kneepkens  CON  3  12.3   4.3  3.0  0.7
107      Olivia Miles  MIN  3  30.3  16.3  3.7  7.0
119       Madina Okot  ATL  2   9.5   4.0  4.5  0.0
128         Kiki Rice  TOR  2  19.0   6.0  2.5  1.0


In [11]:
# Merge predictions with current stats
current_rookies = current_rookies.rename(columns={'Player': 'player'})

validation_df = predictions[['player', 'wnba_team', 'draft_pick', 'predicted_RIS']].merge(
    current_rookies[['player', 'G', 'MP.1', 'PTS', 'TRB', 'AST']],
    on='player',
    how='left'
)

validation_df = validation_df.rename(columns={'MP.1': 'mpg', 'PTS': 'ppg', 
                                               'TRB': 'rpg', 'AST': 'apg'})

print(validation_df[['player', 'predicted_RIS', 'G', 'ppg', 'rpg', 'apg']]
      .sort_values('predicted_RIS', ascending=False)
      .to_string())

              player  predicted_RIS    G   ppg  rpg  apg
7       Lauren Betts          0.595  2.0   3.5  3.5  1.0
8        Madina Okot          0.521  2.0   4.0  4.5  0.0
6          Kiki Rice          0.496  2.0   6.0  2.5  1.0
3   Flau'jae Johnson          0.486  3.0  11.7  4.7  1.7
4    Gabriela Jaquez          0.481  2.0   8.5  6.0  2.0
9       Olivia Miles          0.476  3.0  16.3  3.7  7.0
1          Azzi Fudd          0.414  2.0   5.5  1.0  0.0
10     Raven Johnson          0.333  2.0   2.0  1.5  2.0
11        Taina Mair          0.326  NaN   NaN  NaN  NaN
5   Gianna Kneepkens          0.301  3.0   4.3  3.0  0.7
0     Angela Dugalic          0.300  2.0   3.5  2.0  0.0
2      Cotie McMahon          0.288  NaN   NaN  NaN  NaN


In [13]:
print("\nRoster Notes (as of May 15, 2026):")
print("- Cotie McMahon (WAS, Pick #11): Out for season with UCL tear")
print("- Taina Mair (SEA, Pick #14): Waived before playing a game")
print("- Angela Dugalic (WAS, Pick #9): Now active, stats available")
print("\nThese outcomes illustrate key model limitations:")
print("  injury risk and final roster decisions cannot be predicted from college stats alone")


Roster Notes (as of May 15, 2026):
- Cotie McMahon (WAS, Pick #11): Out for season with UCL tear
- Taina Mair (SEA, Pick #14): Waived before playing a game
- Angela Dugalic (WAS, Pick #9): Now active, stats available

These outcomes illustrate key model limitations:
  injury risk and final roster decisions cannot be predicted from college stats alone


In [14]:
# Save validation snapshot
validation_df['snapshot_date'] = '2026-05-15'
validation_df['games_into_season'] = validation_df['G'].fillna(0).astype(int)

validation_df.to_csv("../data/processed/validation_snapshot.csv", index=False)
print("Saved validation_snapshot.csv!")

Saved validation_snapshot.csv!


In [15]:
from sklearn.preprocessing import MinMaxScaler

# Calculate actual RIS for current rookies
actual_stats = validation_df[validation_df['G'].notna()].copy()

scaler = MinMaxScaler()
actual_scaled = actual_stats[['G', 'mpg', 'ppg', 'rpg', 'apg']].copy()
actual_scaled = pd.DataFrame(scaler.fit_transform(actual_scaled), 
                              columns=['G', 'mpg', 'ppg', 'rpg', 'apg'])

actual_stats['actual_RIS'] = (
    0.25 * actual_scaled['G'] +
    0.25 * actual_scaled['mpg'] +
    0.25 * actual_scaled['ppg'] +
    0.15 * actual_scaled['rpg'] +
    0.10 * actual_scaled['apg']
).round(3)

validation_df = validation_df.merge(actual_stats[['player', 'actual_RIS']], 
                                     on='player', how='left')

# Save updated snapshot with actual RIS
validation_df.to_csv("../data/processed/validation_snapshot.csv", index=False)

print(validation_df[['player', 'predicted_RIS', 'actual_RIS', 'G', 'ppg', 'rpg', 'apg']]
      .sort_values('predicted_RIS', ascending=False).to_string())

              player  predicted_RIS  actual_RIS    G   ppg  rpg  apg
7       Lauren Betts          0.595       0.151  2.0   3.5  3.5  1.0
8        Madina Okot          0.521       0.931  2.0   4.0  4.5  0.0
6          Kiki Rice          0.496       0.190  2.0   6.0  2.5  1.0
3   Flau'jae Johnson          0.486       0.527  3.0  11.7  4.7  1.7
4    Gabriela Jaquez          0.481       0.404  2.0   8.5  6.0  2.0
9       Olivia Miles          0.476       0.044  3.0  16.3  3.7  7.0
1          Azzi Fudd          0.414       0.176  2.0   5.5  1.0  0.0
10     Raven Johnson          0.333         NaN  2.0   2.0  1.5  2.0
11        Taina Mair          0.326         NaN  NaN   NaN  NaN  NaN
5   Gianna Kneepkens          0.301       0.250  3.0   4.3  3.0  0.7
0     Angela Dugalic          0.300       0.091  2.0   3.5  2.0  0.0
2      Cotie McMahon          0.288         NaN  NaN   NaN  NaN  NaN
